In [ ]:
# ===========================================================================
# ACOUSTIC FEATURE ANALYSIS - interactive driver
#
# All extraction/plotting logic lives in src/; this notebook only calls it
# and stores results, matching notebooks/01's convention.
#
#   src/preprocessing.py   MFCC extraction (13 coeff + delta + delta-delta)
#   src/vad.py              Silero VAD (leading/trailing trim, fallback, stats)
#   src/praat.py            per-feature-group extraction (F0, jitter, shimmer,
#                            HNR, CPPS, formants, intensity, speech-rate/pause/
#                            voice-break proxies), framewise segmental/
#                            suprasegmental sequences (GAD voicing decision)
#   src/visualization.py    Praat box plots/summary, feature correlation
#                            heatmap, VAD-validation panel
#   src/eda.py               short-time time/frequency-domain parameters,
#                            wideband/narrowband spectrograms, cepstral
#                            analysis, Linear Prediction analysis, VAD+GAD
#                            three-way silence/unvoiced/voiced segmentation
#   src/style.py              shared color system (SEQUENTIAL_CMAP,
#                            VOICING_COLORS, FORMANT_COLORS, BRANCH_COLORS)
#   src/training/reporting.py  feature_audit() -- per-branch dims/params
#
# Sections:
#   Stage 2-6   Praat statistical EDA (VAD validation, feature extraction,
#               severity-group comparison, correlation, significance testing)
#   Stage 7-15  Speech-processing EDA (production/perception, VAD+GAD
#               segmentation, formant tracks, short-time time/frequency-
#               domain parameters, wideband/narrowband spectrograms,
#               cepstral analysis, MFCC, Linear Prediction analysis)
#   Stage 16-19 Feature-extraction summary (branch inventory, parameter
#               counts, Z_unified composition, channel breakdown)
#
# Features are extracted from the ORIGINAL audio, not the VAD-trimmed/
# zero-padded window the legacy Deep/Acoustic pathways train on - jitter,
# shimmer, HNR, and formants are only meaningful on natural speech (see
# src/praat.py's module docstring for the speech-rate/pause-duration caveat -
# UA-Speech utterances are single isolated words, not continuous speech).
# This table is still required: it's the SHAP-surrogate explainability
# methodology's input (see notebooks/04_model_analysis.ipynb and
# notebooks/06_results.ipynb), independent of which architecture (legacy
# ablation ladder or the three-branch severity model) is being analysed.
# Stage 2 below validates VAD itself (raw vs. VAD-processed waveform/MFCC);
# Praat's own pitch/point-process voicing logic (not VAD) is what decides
# which frames its jitter/shimmer/HNR statistics trust - VAD stats
# (outputs/vad_stats.csv, notebooks/01_data_pipeline.ipynb Stage 9) are
# available only as an independent sanity check, not a second voicing
# decision.
#
# VAD vs GAD: VAD (Silero, src/vad.py) decides speech vs silence at the
# utterance-boundary level. GAD (glottal activity detection -- Praat's
# pitch/point-process voicing decision, src/praat.py) decides, within
# detected speech, which frames are voiced vs unvoiced. Stage 9 below
# combines both into one three-way segmentation, reusing the exact
# voicing computation the Suprasegmental branch trains on.
#
# Color composition: every heatmap (spectrograms, cepstrogram, mel
# filterbank, MFCC) uses the same perceptually-uniform colormap
# (src.style.SEQUENTIAL_CMAP); every silence/unvoiced/voiced panel uses
# the same fixed three-way palette (src.style.VOICING_COLORS); every
# formant overlay uses the same fixed F1/F2/F3 colors
# (src.style.FORMANT_COLORS); every branch figure uses the same fixed
# per-branch colors (src.style.BRANCH_COLORS) -- one consistent visual
# system across every panel in this notebook.
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config, eda
from src.console import print_header, print_kv, architecture_table
from src.preprocessing import (_load_resampled, mfcc_frame_count, build_mfcc_transform,
                               extract_mfcc_features_cached)
from src.praat import extract_suprasegmental_sequence, extract_segmental_extra_sequence
from src.style import apply_style, SEQUENTIAL_CMAP, FORMANT_COLORS, BRANCH_COLORS
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model
from src.training.reporting import feature_audit
from src.vad import apply_vad

apply_style()
config.ensure_directories()

df_m6 = load_manifest()

print_header("Acoustic Feature Analysis")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))


In [ ]:
# STAGE 2 - VAD validation: raw waveform, detected speech region, the
# resulting VAD-processed waveform, and MFCC valid-frames-vs-padding - for
# five examples chosen to each demonstrate a specific VAD behaviour
# (normal duration, long trailing silence, long leading silence, internal
# pauses, weak/low-energy dysarthric). Plotting logic lives in
# src.visualization.plot_vad_validation_examples (moved out of this
# notebook - see that function's docstring for the root-cause explanation
# of what it validates); this cell only calls it.
from src.visualization import plot_vad_validation_examples

vad_validation_path, example_stats = plot_vad_validation_examples(df_m6, seed=config.DEFAULT_SEED)
example_stats


In [ ]:
# STAGE 3 - Extract all FEATURE_COLUMNS Praat features for every utterance in the manifest:
#   pitch        f0 mean/max/min/std/range        (std+range = monopitch)
#   perturbation jitter x4, shimmer x4            (vocal-fold instability)
#   noise        hnr mean/std/min                 (breathiness, roughness)
#   voice qual cpps                             (breathy/dysphonic voice quality)
#   articulation f1/f2/f3 mean+std, f2_f1_ratio   (vowel-space centralization)
#   loudness     intensity mean/max/min/std       (loudness control)
#   rhythm       speech_rate, pause_duration, voice_breaks
#
# Cached to outputs/praat_features.csv - safe to re-run this cell, later runs
# load the cache instead of recomputing ~21k files (~25-30 minutes uncached).
from src.praat import extract_praat_features_batch

praat_features = extract_praat_features_batch(df_m6, cache_path=config.PRAAT_FEATURES_PATH)
praat_features.head()

In [ ]:
# STAGE 4 - Acoustic feature statistics: compare Healthy vs Very Low vs Low vs
# Mid vs High severity groups - box-plot grid for all FEATURE_COLUMNS features,
# saved to outputs/figures/, plus a group-means +/- std table saved to
# outputs/metrics/.
from src.visualization import plot_praat_feature_comparison, build_praat_group_summary

figure_path = plot_praat_feature_comparison(praat_features, show=True)
group_summary = build_praat_group_summary(praat_features)

summary_path = config.METRICS_DIR / "praat_severity_group_summary.csv"
group_summary.to_csv(summary_path)

print_header("Severity Group Comparison")
print_kv("Figure", figure_path)
print_kv("Group summary table", summary_path)
group_summary

In [ ]:
# STAGE 5 - Feature correlation analysis. Which Praat features move together?
# Jitter/shimmer sub-measures and the three formant means are expected to
# correlate strongly since they're different formulas over the same
# underlying signal - this is what tells a reader which features carry
# genuinely independent evidence before Phase 6's Praat pathway treats all
# FEATURE_COLUMNS as independent inputs.
from src.visualization import plot_feature_correlation

correlation_figure = plot_feature_correlation(praat_features, show=True)
print_header("Feature Correlation")
print_kv("Figure", correlation_figure)

In [ ]:
# STAGE 6 - Which of those group differences are actually real?
#
# The box plots above are descriptive; this is the test. Kruskal-Wallis H per
# feature across the five groups (non-parametric, because jitter/shimmer and the
# pause/voice-break proxies are bounded and heavily skewed, so ANOVA's normality
# assumption does not hold), Bonferroni-corrected across the FEATURE_COLUMNS features so that
# testing them all at once does not manufacture significance.
#
# The significant rows are the features the discussion section can legitimately
# claim separate severity levels - and they are the ones worth reading first in
# notebooks/05_error_analysis.ipynb and Phase 6's Praat fusion pathway.
from src.praat import praat_group_significance

significance = praat_group_significance(praat_features)

significance_path = config.METRICS_DIR / "praat_significance.csv"
significance.to_csv(significance_path, index=False)

n_sig = int(significance["significant"].sum())

print_header("Kruskal-Wallis Severity Group Significance")
print_kv("Significance table", significance_path)
print_kv("Significant features (p_adj < 0.05)", f"{n_sig} / {len(significance)}")
print_kv("Strongest separator", significance.iloc[0]["feature"])
significance

In [ ]:
# ===========================================================================
# STAGE 7 - SPEECH-PROCESSING EDA: classical techniques applied directly to
# this project's own corpus (UA-Speech, M6 channel).
#
#   Stage 7    example utterances (Healthy Control + dysarthric High severity)
#   Stage 8    speech production/perception -- source-filter framing
#   Stage 9    VAD + GAD segmentation (silence/unvoiced/voiced)
#   Stage 10   vowel/consonant properties -- formant tracks
#   Stage 11   short-time time-domain parameters (energy, ZCR)
#   Stage 12   short-time frequency-domain parameters (centroid, rolloff)
#   Stage 13   spectrograms -- wideband vs narrowband
#   Stage 14   cepstral analysis
#   Stage 15   MFCC
#   Stage 16   Linear Prediction (LPC) analysis
#
# GMM-HMM vs modern ML/DNN modeling: this project's Segmental/Suprasegmental
# branches use the classical hand-crafted feature family (MFCC, formants,
# HNR, F0, voicing, intensity) that historically paired with a GMM-HMM back
# end, but replace that back end with a small 1D-CNN, since the target here
# is an utterance-level severity label, not a phone/word sequence a HMM's
# temporal alignment is built for. The Learned branch (wav2vec2+LoRA)
# represents the other lineage: a modern self-supervised DNN trained
# end-to-end on raw waveform, no hand-crafted feature stage. The three-branch
# architecture is a direct empirical comparison of both paradigms feeding one
# classifier.
#
# Speech synthesis / E2E applications: out of scope -- this is a
# classification task, not a generative one. The "E2E" idea that IS relevant
# is the Learned branch's own end-to-end feature learning above.
# ===========================================================================
from src.training.data import load_manifest

control_row = df_m6[df_m6["Group"] == "Healthy Control"].sample(1, random_state=config.DEFAULT_SEED).iloc[0]
dysarthric_row = df_m6[df_m6["Severity"] == "High"].sample(1, random_state=config.DEFAULT_SEED).iloc[0]

examples = {"Healthy Control": control_row, "Dysarthric (High severity)": dysarthric_row}
for label, row in examples.items():
    print_kv(label, row["Filename"])


In [ ]:
primary_row = dysarthric_row
waveform_np, sr = _load_resampled(primary_row["Filepath"])
waveform_np = waveform_np.squeeze(0).numpy()

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
t = np.arange(len(waveform_np)) / sr
axes[0].plot(t, waveform_np, color="0.25", linewidth=0.6)
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"{primary_row['Filename']} — waveform")

freqs, times, mag_db = eda.compute_spectrogram(waveform_np, sr, window_ms=25)
im = axes[1].pcolormesh(times, freqs, mag_db, cmap=SEQUENTIAL_CMAP,
                        vmin=mag_db.max() - 80, vmax=mag_db.max(), shading="auto")
axes[1].set_ylim(0, 5000)
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Spectrogram (25 ms window)")
fig.colorbar(im, ax=axes, label="dB", fraction=0.03, pad=0.02)
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda01_production_overview.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
from src.praat import extract_suprasegmental_sequence

total_frames = mfcc_frame_count(len(waveform_np))
supra_seq = extract_suprasegmental_sequence(waveform_np, sr, valid_length=len(waveform_np),
                                            total_frames=total_frames)
voicing_mask = supra_seq["voicing"]

_, vad_stats = apply_vad(torch.from_numpy(waveform_np).unsqueeze(0), sr=sr)
speech_start_s = vad_stats["leading_trimmed_s"]
speech_end_s = vad_stats["original_duration_s"] - vad_stats["trailing_trimmed_s"]
print_kv("VAD speech span", f"{speech_start_s:.3f}s - {speech_end_s:.3f}s "
        f"(of {vad_stats['original_duration_s']:.3f}s)")

labels = eda.three_way_segmentation(total_frames, frame_hop_s=0.01,
                                    speech_start_s=speech_start_s, speech_end_s=speech_end_s,
                                    voicing_mask=voicing_mask)
ste_t, ste_v = eda.short_time_energy(waveform_np, sr)
zcr_t, zcr_v = eda.zero_crossing_rate(waveform_np, sr)

eda.plot_voiced_unvoiced_silence(waveform_np, sr, labels, frame_hop_s=0.01,
                                 title=f"{primary_row['Filename']} — VAD + GAD segmentation",
                                 ste=(ste_t, ste_v), zcr=(zcr_t, zcr_v), show=True,
                                 out_name="eda02_voiced_unvoiced_silence.png")

voiced_pct = (labels == 2).mean() * 100
unvoiced_pct = (labels == 1).mean() * 100
silence_pct = (labels == 0).mean() * 100
print_kv("Silence / unvoiced / voiced", f"{silence_pct:.1f}% / {unvoiced_pct:.1f}% / {voiced_pct:.1f}%")


In [ ]:
from src.praat import extract_segmental_extra_sequence

vad_waveform, vad_stats2 = apply_vad(torch.from_numpy(waveform_np).unsqueeze(0), sr=sr)
vad_waveform_np = vad_waveform.squeeze(0).numpy()
seg_total_frames = mfcc_frame_count(len(vad_waveform_np))
extra_seq = extract_segmental_extra_sequence(vad_waveform_np, sr, valid_length=len(vad_waveform_np),
                                             total_frames=seg_total_frames)

freqs_n, times_n, mag_db_n = eda.compute_spectrogram(vad_waveform_np, sr, window_ms=30)
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.pcolormesh(times_n, freqs_n, mag_db_n, cmap=SEQUENTIAL_CMAP,
                   vmin=mag_db_n.max() - 80, vmax=mag_db_n.max(), shading="auto")
ax.set_ylim(0, 4000)

frame_times = np.arange(seg_total_frames) * 0.01
for name, arr in (("F1", extra_seq["f1_hz"]), ("F2", extra_seq["f2_hz"]), ("F3", extra_seq["f3_hz"])):
    valid = arr > 0
    ax.scatter(frame_times[valid], arr[valid], s=6, color=FORMANT_COLORS[name], label=name)

ax.legend(loc="upper right", fontsize=9)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"{primary_row['Filename']} — formant tracks on narrowband spectrogram")
fig.colorbar(im, ax=ax, label="dB", fraction=0.03, pad=0.02)
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda03_formant_tracks.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex="col")
for col, (label, row) in enumerate(examples.items()):
    wav, sr_ex = _load_resampled(row["Filepath"])
    wav = wav.squeeze(0).numpy()
    t_ste, v_ste = eda.short_time_energy(wav, sr_ex)
    t_zcr, v_zcr = eda.zero_crossing_rate(wav, sr_ex)
    axes[0, col].plot(t_ste, v_ste, color="#4c72b0")
    axes[0, col].set_title(label)
    axes[1, col].plot(t_zcr, v_zcr, color="#dd8452")
axes[0, 0].set_ylabel("Short-time energy")
axes[1, 0].set_ylabel("Zero-crossing rate")
for ax in axes[1, :]:
    ax.set_xlabel("Time (s)")
fig.suptitle("Short-time time-domain parameters — Control vs. Dysarthric")
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda04_time_domain_parameters.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
t_sc, centroid, rolloff = eda.spectral_centroid_rolloff(waveform_np, sr)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_sc, centroid, color="#55a868", label="Spectral centroid")
ax.plot(t_sc, rolloff, color="#c44e52", linestyle="--", label="Spectral rolloff (85%)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"{primary_row['Filename']} — short-time frequency-domain parameters")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda05_frequency_domain_parameters.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
eda.plot_wideband_narrowband_spectrogram(vad_waveform_np, sr,
                                          title=f"{primary_row['Filename']} — wideband vs. narrowband",
                                          show=True, out_name="eda06_wideband_narrowband.png")


In [ ]:
voiced_frame_times = frame_times[extra_seq["f1_hz"] > 0]
frame_center = float(voiced_frame_times[len(voiced_frame_times) // 2]) if len(voiced_frame_times) else 0.2

eda.plot_cepstral_analysis(vad_waveform_np, sr, frame_center_s=frame_center,
                           title=f"{primary_row['Filename']} — cepstral analysis at t={frame_center:.2f}s",
                           show=True, out_name="eda07_cepstral_analysis.png")


In [ ]:
from src.preprocessing import extract_mfcc_features_cached

mfcc_tensor = extract_mfcc_features_cached(primary_row["Filepath"]).squeeze(0)  # (39, T): MFCC+delta+delta-delta
mfcc_np = mfcc_tensor[:config.N_MFCC].numpy()  # static coefficients only, for the classic MFCC heatmap

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(mfcc_np, aspect="auto", origin="lower", cmap=SEQUENTIAL_CMAP,
               extent=[0, mfcc_np.shape[1] * 0.01, 0, config.N_MFCC])
ax.set_xlabel("Time (s)")
ax.set_ylabel("MFCC coefficient")
ax.set_title(f"{primary_row['Filename']} — MFCC ({config.N_MFCC} coefficients)")
fig.colorbar(im, ax=ax, label="Coefficient value", fraction=0.03, pad=0.02)
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda08_mfcc.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
eda.plot_lpc_envelope(vad_waveform_np, sr, frame_center_s=frame_center, order=16,
                      title=f"{primary_row['Filename']} — LPC envelope (order 16) at t={frame_center:.2f}s",
                      show=True, out_name="eda09_lpc_envelope.png")


In [ ]:
# ===========================================================================
# STAGE 17 - FEATURE-EXTRACTION SUMMARY: formal inventory of what each
# branch of src/models/gated_fusion.py's GatedFusionModel extracts --
# input representation, channel/dimension counts, trainable-parameter
# budget. No training happens here; every number is read off a real
# (untrained) model instance and src/config.py's centralized dimensions
# via src.training.reporting.feature_audit / src.console.architecture_table
# -- nothing here is retyped by hand.
# ===========================================================================
model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES["severity"], num_speakers=2)
audit = feature_audit(model=model, num_classes=config.NUM_CLASSES["severity"])

print_header("Feature-extraction summary")
print_kv("Model", SEVERITY_MODEL_NAME)
print_kv("Branches", 3)
print_kv("Fused representation", f"{audit['fusion']['fused_dim']}-dim")


In [ ]:
branch_inventory = pd.DataFrame([
    {
        "branch": "Learned",
        "input_representation": "Raw waveform (16 kHz, 4.0 s window)",
        "extraction_method": f"{audit['learned_branch']['wav2vec2_model']} "
                             f"+ LoRA (rank {audit['learned_branch']['lora_rank']}, "
                             f"alpha {audit['learned_branch']['lora_alpha']}) on "
                             f"{', '.join(audit['learned_branch']['lora_target_modules'])}",
        "raw_dim": config.WAV2VEC_EMBED_DIM,
        "bottleneck_dim": audit["learned_branch"]["dimensions"],
        "source_module": "src/models/deep_pathway.py (DeepPathway)",
        "trainable": "Partial — LoRA adapters + projection only",
    },
    {
        "branch": "Segmental",
        "input_representation": "MFCC + \u0394 + \u0394\u0394 + framewise formants (F1-F3) + framewise HNR",
        "extraction_method": "3-layer 1D-CNN + masked mean-pool",
        "raw_dim": audit["segmental_branch"]["input_channels"],
        "bottleneck_dim": audit["segmental_branch"]["dimensions"],
        "source_module": "src/models/segmental_pathway.py (SegmentalPathway)",
        "trainable": "Fully trainable (CNN + bottleneck)",
    },
    {
        "branch": "Suprasegmental",
        "input_representation": "F0 (semitones) + voicing mask + intensity (dB)",
        "extraction_method": "2-layer 1D-CNN + masked mean-pool",
        "raw_dim": audit["suprasegmental_branch"]["input_channels"],
        "bottleneck_dim": audit["suprasegmental_branch"]["dimensions"],
        "source_module": "src/models/suprasegmental_pathway.py (SuprasegmentalPathway)",
        "trainable": "Fully trainable (CNN + bottleneck)",
    },
])
branch_inventory


In [ ]:
branch_inventory.to_csv(config.TABLES_DIR / "feature_branch_inventory.csv", index=False)
print_kv("Saved", config.TABLES_DIR / "feature_branch_inventory.csv")


In [ ]:
param_table = architecture_table(model, model_name=SEVERITY_MODEL_NAME)
param_table_display = param_table.copy()
for c in ("trainable_params", "total_params", "frozen_params"):
    param_table_display[c] = param_table_display[c].map(lambda v: f"{v:,}")
param_table_display["pct_trainable"] = param_table_display["pct_trainable"].map(lambda v: f"{v:.2f}%")
param_table_display


In [ ]:
param_table.to_csv(config.TABLES_DIR / "feature_branch_parameter_counts.csv", index=False)
print_kv("Saved", config.TABLES_DIR / "feature_branch_parameter_counts.csv")


In [ ]:
import matplotlib.pyplot as plt

fused = audit["fusion"]
branches = ["learned", "segmental", "supra"]
dims = [fused["learned_dim"], fused["segmental_dim"], fused["supra_dim"]]
labels = ["Learned\n(wav2vec2+LoRA)", "Segmental\n(MFCC+formants+HNR)", "Suprasegmental\n(F0+voicing+intensity)"]
colors = [BRANCH_COLORS[b] for b in branches]

fig, ax = plt.subplots(figsize=(7, 2.2))
left = 0
for dim, label, color in zip(dims, labels, colors):
    ax.barh(0, dim, left=left, color=color, edgecolor="white", height=0.6)
    ax.text(left + dim / 2, 0, f"{label}\n{dim}D", ha="center", va="center", fontsize=8.5, color="white")
    left += dim
ax.set_xlim(0, fused["fused_dim"])
ax.set_yticks([])
ax.set_xlabel(f"Z_unified dimension (total {fused['fused_dim']}D)")
ax.set_title("Fused embedding composition")
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "feature_zunified_composition.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
families = audit["segmental_branch"]["engineered_feature_families"]
family_names, family_counts = [], []
for entry in families:
    name, count_str = entry.rsplit("(", 1)
    family_names.append(name.strip())
    family_counts.append(int(count_str.rstrip(")")))

assert sum(family_counts) == audit["segmental_branch"]["input_channels"]

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = plt.get_cmap("viridis")([i / max(1, len(family_names) - 1) for i in range(len(family_names))])
bars = ax.bar(family_names, family_counts, color=bar_colors, edgecolor="white")
ax.bar_label(bars, padding=2)
ax.set_ylabel("Channels")
ax.set_title(f"Segmental branch input channels (total {sum(family_counts)})")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "feature_segmental_channel_breakdown.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
supra_families = audit["suprasegmental_branch"]["engineered_feature_families"]
supra_table = pd.DataFrame({"channel": range(1, len(supra_families) + 1), "feature": supra_families})
supra_table
